In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [2]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [136]:
os.chdir("/home/lishaogang/NEU2026-Jan")

In [95]:
adata_B = sc.read_h5ad('adata_B_raw.h5ad')

In [96]:
# import scanpy as sc
# import numpy as np

# # 1. 过滤掉 T 细胞数量太少的样本（样本内至少要有 20 个 T 细胞才参与 HVG 计算）
# min_cell_for_hvg = 20
# cell_counts = adata_B.obs['orig.ident'].value_counts()
# valid_samples = cell_counts[cell_counts >= min_cell_for_hvg].index



# # 剔除那些只有零星几个 B 细胞的样本
# adata_B = adata_B[adata_B.obs['orig.ident'].isin(valid_samples)].copy()
# print(f"--- 过滤后剩余样本数: {len(valid_samples)} ---")

# print(f"--- 过滤后剩余 B 细胞数: {adata_B.n_obs} ---")


In [97]:
# 2. 重新过滤基因（只保留在当前 T 细胞子集中有表达的基因）
sc.pp.filter_genes(adata_B, min_cells=3)
# 3. 运行 HVG
print("\n--- Finding HVGs within the B-cell subset ---")
sc.pp.highly_variable_genes(adata_B, batch_key='dataset', n_top_genes=2000, flavor='seurat')


# 4. 提取 HVG 并后续处理
adata_B_hvg = adata_B[:, adata_B.var.highly_variable].copy()
sc.pp.scale(adata_B_hvg, zero_center=False)
sc.tl.pca(adata_B_hvg, n_comps=50, random_state=seed)

print("✅ PCA 计算完成！")

In [99]:
# --- Drawing a Scree Plot ---
print("Drawing a scree plot to assist in determining the selection of n_pcs...")

# 1. Modify Matplotlib's global default figure size be# 计算邻居
sc.pp.neighbors(adata_B, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')

# 计算 UMAP
sc.tl.umap(adata_B)

# 重新聚类 (因为删除了簇4，建议重新聚类一下看看剩下的细胞怎么分)
sc.tl.leiden(adata_B, resolution=0.5, key_added='leiden_B')

# 可视化
sc.pl.umap(adata_B, color=['leiden_B', 'dataset'])
#    This will affect all figures created afterwards
plt.rcParams['figure.figsize'] = (8, 6)  # Set to 8x6 inches

# 2. Now directly call scanpy's plotting function without any additional parameters
#    When it internally creates the figure, it will follow the global size set above
sc.pl.pca_variance_ratio(adata_B_hvg, log=True, n_pcs=50)
plt.show()

# 3. Restore Matplotlib's default settings
#    This is a good practice to prevent this size setting from affecting your subsequent plots
plt.rcParams.update(plt.rcParamsDefault)

In [100]:
# --- 5. 运行 Harmony (校正批次效应) ---
import scanpy.external as sce

print("\n--- 正在为 B 亚群运行 Harmony ---")
sce.pp.harmony_integrate(
    adata_B_hvg, 
    key='orig.ident', 
    basis='X_pca', 
    adjusted_basis='X_pca_harmony'
)

# --- 6. 映射结果并聚类 ---
adata_B.obsm['X_pca_harmony'] = adata_B_hvg.obsm['X_pca_harmony']

# 计算邻居和 UMAP (建议将邻居数调小一点，比如 10-15，因为 NK 子集通常比较小)
sc.pp.neighbors(adata_B, n_neighbors=15,n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_B)


In [103]:
sc.tl.leiden(adata_B, resolution=0.4, key_added='leiden_B')

# --- 7. 可视化 ---
sc.pl.umap(adata_B, color='leiden_B', wspace=0.4)

In [104]:
# --- 7. 可视化 ---
sc.pl.umap(adata_B, color='leiden_B',legend_loc = 'on data')

In [105]:
# 从B细胞亚群文本中提取的所有基因名称列表
gene_list = ['CD3D','CD3E','CD3G',
    'CD79A', 'CD19', 'MS4A1', 'NEIL1', 'CD9', 'CD38', 'TCL1A', 'SOX4', 'MME',
    'CD27', 'IGHD', 'IL4R', 'FCER2', 'BACH2', 'IGHA1', 'ITGAX', 'FCRL5',
    'CD1C', 'SDC1', 'JCHAIN', 'XBP1', 'MZB1', 'TNFRSF17', 'MKI67'
]


# --- Check for marker gene availability in the data ---
# Use .raw.var_names as it contains all genes before HVG filtering.
available_genes_in_raw = adata_B.raw.var_names.tolist()
genes_to_plot = [gene for gene in gene_list if gene in available_genes_in_raw]

skipped_genes = set(gene_list) - set(genes_to_plot)
if skipped_genes:
    print(f"Warning: The following genes were not found in the data and will be skipped:\n{list(skipped_genes)}")




In [106]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata_B,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_B",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata_B,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_B",
    standard_scale='var',
    cmap='Reds',
    use_raw=True,
    figsize=(12, 6) # 在这里直接手动指定 宽度和高度
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)





In [107]:
adata_B = adata_B[adata_B.obs['leiden_B'] != '4', :].copy()

In [108]:
# 查看筛选后leiden_B的剩余标签
print("筛选后剩余聚类标签：", adata_B.obs['leiden_B'].unique())
# 查看细胞数量变化（确认T_doublet细胞已删除）
print("筛选前细胞数：", adata_B.shape[0])  # 筛选后运行此句，对比原始数量

In [110]:
# 就是这一行！把旧结果里属于现在这些细胞的坐标提出来
adata_B.obsm['X_pca_harmony'] = adata_B_hvg[adata_B.obs_names, :].obsm['X_pca_harmony']

# --- 现在你可以放心地运行你那几步了 ---
sc.pp.neighbors(adata_B, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_B)
sc.tl.leiden(adata_B, resolution=0.5, key_added='leiden_B')
sc.pl.umap(adata_B, color=['leiden_B', 'dataset'])

In [111]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata_B,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_B",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata_B,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_B",
    standard_scale='var',
    cmap='Reds',
    use_raw=True,
    figsize=(12, 6) # 在这里直接手动指定 宽度和高度
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)





In [112]:


annotation_dict = {
    "0": "naive B cell",
    "1": "naive B cell",
    "2": "memory B cell",
    "3": "memory B cell",
    "4": "memory B cell",
    "5": "memory B cell",
    "6": "plasma cell",
    "7": "memory B cell",
    "8": "naive B cell",
    "9": "naive B cell",
    "10": "transitional B cell",
    "11": "memory B cell"
}

In [113]:


# 创建一个新的 obs 列来存储精细化的注释
adata_B.obs['cell_subtype'] = adata_B.obs['leiden_B'].map(annotation_dict).astype('category')





# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(10,5))  

# 第一个面板：用leiden_refined_1.5作为color，指定到axes[0]
sc.pl.umap(adata_B,       
           color='leiden_B',
           legend_loc='on data', 
           title='leiden_refined Source',  # 区分标题   
           ax=axes[0],  # 第一个子图位置
           frameon=False, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata_B, 
           color='cell_subtype',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=False, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [114]:
# 1. 剔除第 7 簇
print(f"剔除前细胞数: {adata_B.n_obs}")
adata_B = adata_B[adata_B.obs['leiden_B'] != '7', :].copy()
print(f"剔除后细胞数: {adata_B.n_obs}")

# 2. 强制对齐旧的 Harmony 坐标 (防止报错)
# 这一步是为了不重跑 Harmony 也能直接画图
adata_B.obsm['X_pca_harmony'] = adata_B_hvg[adata_B.obs_names, :].obsm['X_pca_harmony']

# 3. 重新计算 UMAP（删除了污染簇后，UMAP 布局会更清晰）
sc.pp.neighbors(adata_B, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_B)


In [115]:
sc.tl.leiden(adata_B, resolution=0.5, key_added='leiden_B')
# 4. 可视化查看清理后的结果
sc.pl.umap(adata_B, color=['leiden_B', 'dataset'], title='B-cell Subset (Cleaned)')

In [117]:
sc.pl.umap(adata_B, color='leiden_B', title='B-cell Subset (Cleaned)',legend_loc = 'on data')

In [118]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata_B,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_B",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata_B,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_B",
    standard_scale='var',
    cmap='Reds',
    use_raw=True,
    figsize=(12, 6) # 在这里直接手动指定 宽度和高度
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)





In [123]:
annotation_dict = {
    "0": "Naive B cell",
    "1": "Naive B cell",
    "2": "Memory B cell",
    "3": "Memory B cell",
    "4": "Memory B cell",
    "5": "Plasma cell",
    "6": "Naive B cell",
    "7": "Naive B cell",
    "8": "Transitional B cell",
    "9": "Memory B cell"
}

In [125]:


# 创建一个新的 obs 列来存储精细化的注释
adata_B.obs['cell_subtype'] = adata_B.obs['leiden_B'].map(annotation_dict).astype('category')





# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(10,5))  

# 第一个面板：用leiden_refined_1.5作为color，指定到axes[0]
sc.pl.umap(adata_B,       
           color='leiden_B',
           legend_loc='on data', 
           title='leiden_refined Source',  # 区分标题   
           ax=axes[0],  # 第一个子图位置
           frameon=False, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata_B, 
           color='cell_subtype',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=False, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [137]:
import scanpy as sc
import pandas as pd
adata_B = sc.read_h5ad("adata_B_final.h5ad")


In [153]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

# 0. 禁止滚动条
display(HTML("<style>.output_scroll { height: auto !important; max-height: none !important; }</style>"))

# ========================
# 1. 强制重新映射（确保名称万无一失）
# ========================
# 重新执行你的映射逻辑，确保 adata_B.obs['cell_subtype'] 是干净的
annotation_dict = {
    "0": "Naive B cell", "1": "Naive B cell", "6": "Naive B cell", "7": "Naive B cell",
    "2": "Memory B cell", "3": "Memory B cell", "4": "Memory B cell", "9": "Memory B cell",
    "5": "Plasma cell",
    "8": "Transitional B cell"
}
adata_B.obs['cell_subtype'] = adata_B.obs['leiden_B'].map(annotation_dict).astype('category')
adata_B.obs['cell_subtype'] = adata_B.obs['cell_subtype'].cat.remove_unused_categories()

# ========================
# 2. 自动适配基因名 (解决大小写问题)
# ========================
# 获取当前数据集中所有的基因名
all_genes = adata_B.var_names.tolist()
if adata_B.raw is not None:
    all_genes = list(set(all_genes + adata_B.raw.var_names.tolist()))

def get_real_gene(gene_list):
    """自动匹配数据中真实的基因名（不区分大小写）"""
    real_genes = []
    for g in gene_list:
        # 尝试全大写、首字母大写、全小写
        for potential in [g.upper(), g.capitalize(), g.lower()]:
            if potential in all_genes:
                real_genes.append(potential)
                break
    return list(dict.fromkeys(real_genes)) # 去重

# 定义原始字典
raw_marker_dict = {
     "Pan-B": ["CD19", "MS4A1"], 
    "Naive B cell": [ "IGHD", "TCL1A", "FCER2", "BACH2"],
    "Transitional B cell": ["SOX4", "MME", "NEIL1",'TCL1A','SOX4'],


    "Memory B cell": ["CD27","IGHA1"],
    "Plasma cell": ["CD38", "SDC1", "JCHAIN", "XBP1", "MZB1", "TNFRSF17"]
}

# 转换成数据中真实的基因名
valid_markers = {k: get_real_gene(v) for k, v in raw_marker_dict.items()}

# 检查是否找到了基因
total_found = sum(len(v) for v in valid_markers.values())
if total_found == 0:
    print("❌ 错误：在数据中未找到任何 Marker 基因！请检查 var_names。")
    print(f"数据前5个基因示例：{all_genes[:5]}")
else:
    print(f"✅ 成功找到 {total_found} 个匹配的基因。")



In [154]:
# ========================
# 3. 设置排序与绘图
# ========================
final_order = ["Naive B cell","Transitional B cell",  "Memory B cell", "Plasma cell"]
# 过滤掉数据中万一不存在的亚型
final_order = [s for s in final_order if s in adata_B.obs['cell_subtype'].cat.categories]
adata_B.obs['cell_subtype'] = adata_B.obs['cell_subtype'].cat.reorder_categories(final_order)

# 自动决定是否使用 raw（如果 .raw 存在且包含 Marker，则使用）
use_raw_setting = True if (adata_B.raw is not None) else False

# 绘图
sc.set_figure_params(dpi_save=300, frameon=False, vector_friendly=True)
try:
    # 直接调用，不使用 make_figure 可能会避开某些底层报错
    sc.pl.dotplot(
        adata_B,
        var_names=valid_markers,
        groupby="cell_subtype",
        use_raw=use_raw_setting,
        standard_scale='var',
        cmap='Reds',
        figsize=(max(10, total_found*0.6), 3.5),
        show=True,
        save='_B_cell_staircase.png' # 自动保存到当前目录的 figures 文件夹
    )
except Exception as e:
    print(f"❌ 绘图依然失败，原因: {e}")
    print("尝试关闭 standard_scale 重试...")
    sc.pl.dotplot(adata_B, valid_markers, groupby="cell_subtype", use_raw=use_raw_setting)

In [133]:
sc.pl.umap(adata_B, 
           color='cell_subtype',
           legend_loc='on data', 
           title='cell_type Source',
           frameon=False)

In [ ]:
#保存adata_B
adata_B.write_h5ad("adata_B_final.h5ad")